# Lecture 3 · The Debug Ladder

Walk through all 7 rungs of Karpathy's debugging ladder on a real classification task.

Every rung is a **sanity check**. If a rung fails, fix it before climbing. If you skip a rung you will chase a bug you could have caught in 30 seconds.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Rung 1 · Inspect the data

Before looking at the model, understand the data. A toy binary-classification task with 2D inputs.

In [ ]:
def make_two_moons(n=1000, noise=0.1):
    n_half = n // 2
    t = np.linspace(0, np.pi, n_half)
    X0 = np.stack([np.cos(t), np.sin(t)], axis=1) + noise * np.random.randn(n_half, 2)
    X1 = np.stack([1 - np.cos(t), 1 - np.sin(t) - 0.5], axis=1) + noise * np.random.randn(n_half, 2)
    X = np.vstack([X0, X1]).astype(np.float32)
    y = np.hstack([np.zeros(n_half), np.ones(n_half)]).astype(np.int64)
    return X, y

X, y = make_two_moons(n=2000)

print(f'X shape: {X.shape}   dtype: {X.dtype}   range: [{X.min():.2f}, {X.max():.2f}]')
print(f'y shape: {y.shape}   dtype: {y.dtype}   classes: {np.bincount(y)}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[y==0, 0], X[y==0, 1], c='#7b9e89', s=8, label='class 0')
ax.scatter(X[y==1, 0], X[y==1, 1], c='#d97757', s=8, label='class 1')
ax.set_title('two moons')
ax.legend()
plt.show()

**Check** · shapes correct, label balance ~50/50, no NaN or outlier in the ranges, plot matches expectation.

## Rung 2 · Dumb baseline

How good is 'most frequent class'? Any model must beat this.

In [ ]:
baseline_acc = (y == y.mean().round()).mean()
print(f'majority-class accuracy: {baseline_acc:.3f}')

## Rung 3 · Overfit one batch

The single most useful diagnostic. Train on 8 examples and confirm the loss reaches ≈ 0. If not, your model/loss/optimizer has a bug.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x): return self.net(x)

model = MLP().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

# Tiny batch
xb = torch.from_numpy(X[:8]).to(device)
yb = torch.from_numpy(y[:8]).to(device)

losses = []
for step in range(300):
    logits = model(xb)
    loss = criterion(logits, yb)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

plt.plot(losses)
plt.yscale('log')
plt.xlabel('step'); plt.ylabel('loss (log)')
plt.title(f'overfit-one-batch · final loss = {losses[-1]:.2e}')
plt.show()

assert losses[-1] < 1e-3, 'Loss should reach ≈ 0 on 8 examples — otherwise there is a bug.'

**Passed!** The loss reaches ≈ 0. The architecture, loss, and optimizer are wired correctly. If it *didn't*, check the L3 slide's checklist · dead ReLUs, wrong label shape, missing zero_grad, etc.

## Rung 4 · Small subset

Train on 5% of the data. Confirm the full training pipeline works end-to-end.

In [ ]:
def train(X, y, n_epochs=20, lr=1e-3, batch_size=64, verbose=False):
    Xt = torch.from_numpy(X).to(device)
    yt = torch.from_numpy(y).to(device)
    ds = TensorDataset(Xt, yt)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model = MLP().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    losses = []
    for ep in range(n_epochs):
        for xb, yb in loader:
            logits = model(xb)
            loss = nn.functional.cross_entropy(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        if verbose and ep % 5 == 0:
            with torch.no_grad():
                acc = (model(Xt).argmax(-1) == yt).float().mean().item()
            print(f'epoch {ep:2d}  loss={loss.item():.4f}  acc={acc:.3f}')
    return model, losses

# Train on 5% of data
idx = np.random.permutation(len(X))[:len(X)//20]
X_sub, y_sub = X[idx], y[idx]
print(f'using {len(X_sub)} of {len(X)} examples')

model_sub, losses_sub = train(X_sub, y_sub, n_epochs=30, verbose=True)
plt.plot(losses_sub); plt.yscale('log'); plt.title('5% subset · training loss'); plt.show()

## Rung 5 · LR finder

Sweep the learning rate geometrically; find the range where loss falls fastest.

In [ ]:
def lr_finder(X, y, lr_start=1e-7, lr_end=10, n_steps=100):
    Xt = torch.from_numpy(X).to(device)
    yt = torch.from_numpy(y).to(device)
    model = MLP().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr_start)

    lrs, losses = [], []
    gamma = (lr_end / lr_start) ** (1 / n_steps)

    for i in range(n_steps):
        # Pick random batch
        idx = torch.randint(0, len(Xt), (64,))
        xb, yb = Xt[idx], yt[idx]
        logits = model(xb)
        loss = nn.functional.cross_entropy(logits, yb)
        if torch.isnan(loss) or loss.item() > 10:
            break

        lrs.append(opt.param_groups[0]['lr'])
        losses.append(loss.item())

        opt.zero_grad(); loss.backward(); opt.step()

        # Grow LR
        for g in opt.param_groups:
            g['lr'] *= gamma
    return lrs, losses

lrs, losses = lr_finder(X, y)
plt.semilogx(lrs, losses)
plt.xlabel('learning rate'); plt.ylabel('loss')
plt.title('LR finder — pick ~1 decade before the minimum')
plt.grid(True, alpha=0.3)
plt.show()

Pick the LR at the **steepest descent** (typically around the minimum of the curve). Rule of thumb · pick one decade before the minimum for a safety margin.

## Rung 6 · Full run + regularization

Now train on all the data with a reasonable LR. Watch train/val loss for overfitting.

In [ ]:
# Split train/val
split = int(0.8 * len(X))
X_tr, y_tr = X[:split], y[:split]
X_va, y_va = X[split:], y[split:]

model, tr_losses = train(X_tr, y_tr, n_epochs=30, lr=1e-2)

with torch.no_grad():
    pred_va = model(torch.from_numpy(X_va).to(device)).argmax(-1).cpu().numpy()
    pred_tr = model(torch.from_numpy(X_tr).to(device)).argmax(-1).cpu().numpy()

print(f'train acc: {(pred_tr == y_tr).mean():.3f}')
print(f'val acc:   {(pred_va == y_va).mean():.3f}')

## Rung 7 · Error analysis

Where does the model fail? Look at the val mistakes visually.

In [ ]:
# Decision boundary plot
grid_x, grid_y = np.meshgrid(np.linspace(-1.5, 2.5, 100), np.linspace(-1.5, 1.5, 100))
grid = np.stack([grid_x.ravel(), grid_y.ravel()], axis=1).astype(np.float32)
with torch.no_grad():
    probs = F.softmax(model(torch.from_numpy(grid).to(device)), -1)[:, 1].cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(grid_x, grid_y, probs.reshape(grid_x.shape), levels=20, alpha=0.4, cmap='RdYlGn_r')
correct = pred_va == y_va
ax.scatter(X_va[correct, 0],  X_va[correct, 1],  c='k', s=10, label='correct')
ax.scatter(X_va[~correct, 0], X_va[~correct, 1], c='red', s=30, marker='x', label='wrong')
ax.set_title(f'decision boundary · {(~correct).sum()} val errors')
ax.legend()
plt.show()

Observe · errors concentrate at the **boundary between the two moons**. To improve, you could:
- Add more capacity
- Use a wider kernel / different architecture
- Collect more boundary examples

That's **error analysis** — look at where the model fails; let failures drive the next iteration.

---

**Reflection** · every rung caught a potential bug (or confirmed sanity). Never skip a rung.